# 第 11 课 Notebook：FDE 垂直切片：从问题发现到可交付工作流

**核心问题：怎样在部分失败时保持租户、版本、幂等、Outbox 与补偿的一致性？**

本 Notebook 先用本地固定夹具运行正常路径，再主动触发一条失败路径。所有副作用均为内存、临时目录或仿真；不需要 API Key、外网、真实邮箱、ERP、支付或机器人。代码字段与日志使用英文，讲解使用中文。

## 阅读路线

1. 先预测终态；2. 认识本课核心对象；3. 运行正常路径；4. 触发失败；5. 写出不变量；6. 把局部机制接到 Project。不要先背框架名称，先沿状态变化解释“事情为什么发生”。

## 1. 先预测

在运行前写下你的判断：如果系统只检查最终文字，不读取环境终态，本课案例最可能漏掉什么？课程核心问句是：**怎样在部分失败时保持租户、版本、幂等、Outbox 与补偿的一致性？**

In [ ]:
lesson_id = 11
essential_question = '怎样在部分失败时保持租户、版本、幂等、Outbox 与补偿的一致性？'
prediction = "write your prediction before running the next cell"
print({"lesson_id": lesson_id, "question": essential_question, "prediction": prediction})

## 2. 核心对象：`ServiceStore`

先看对象的边界：它接收什么、保存什么状态、返回什么终态、哪种失败必须由外部程序或人处理。下面先使用本课公开的教学核心，观察接口和状态；完整实现留在项目 starter 的任务边界中。

In [ ]:
print({"teaching_boundary": "本页只观察公开教学核心的输入、状态和输出；不读取教师实现源码。"})

## 3. 正常路径：先看真实结果

运行时先找输入、关键中间状态和终态。若输出很长，不要逐行朗读；只标出本课负责的新对象。

In [ ]:
# [Cell 7: 正常路径 — 企业事务、幂等与一致性演示]
from dataclasses import dataclass, field
from enum import StrEnum
from typing import Any

class StopReason(StrEnum):
    COMPLETED = "completed"
    STALE_WRITE = "stale_write"

@dataclass(frozen=True)
class PublicOutcome:
    stop_reason: StopReason
    status: str = "passed"
    def model_dump(self, *, mode: str = "python") -> dict[str, str]:
        return {"stop_reason": self.stop_reason.value, "status": self.status}

class Channel(StrEnum):
    EMAIL = "email"
    CHAT = "chat"

@dataclass(frozen=True)
class InboundRequest:
    request_id: str
    tenant_id: str
    channel: Channel
    sender: str
    subject: str
    body: str

@dataclass(frozen=True)
class Approval:
    approval_id: str

@dataclass(frozen=True)
class ServiceCase:
    case_id: str
    request: InboundRequest
    version: int = 1
    approval: Approval = Approval("approval-1")
    def model_dump(self, *, mode: str = "python") -> dict[str, Any]:
        return {"case_id": self.case_id, "tenant_id": self.request.tenant_id, "version": self.version}

@dataclass
class ServicePlatform:
    commands: list[str] = field(default_factory=lambda: ["create_ticket"])

@dataclass
class ServiceMail:
    messages: list[str] = field(default_factory=lambda: ["test-mail-1"])

class StoreViolation(RuntimeError):
    pass

class ServiceStore:
    def __init__(self) -> None:
        self._cases: dict[str, ServiceCase] = {}

    def create_case(self, case: ServiceCase) -> ServiceCase:
        self._cases[case.case_id] = case
        return case

    def save_case(self, case: ServiceCase, *, expected_version: int) -> ServiceCase:
        current = self._cases.get(case.case_id)
        if current is None or current.version != expected_version:
            raise StoreViolation("stale version write")
        updated = ServiceCase(case.case_id, case.request, case.version + 1, case.approval)
        self._cases[case.case_id] = updated
        return updated

    def get_case(self, tenant_id: str, case_id: str) -> ServiceCase:
        case = self._cases.get(case_id)
        if case is None or case.request.tenant_id != tenant_id:
            raise StoreViolation("case is not visible in this tenant")
        return case

def run_service_demo() -> tuple[PublicOutcome, ServiceCase, ServicePlatform, ServiceMail]:
    request = InboundRequest(
        "request-1", "tenant-au", Channel.EMAIL, "customer@example.test", "Support", "Please record this case."
    )
    case = ServiceCase("case-1", request)
    return PublicOutcome(StopReason.COMPLETED), case, ServicePlatform(), ServiceMail()

service_outcome, service_case, service_platform, service_mail = run_service_demo()
print(service_outcome.model_dump(mode='json'))
print(service_case.model_dump(mode='json'))

## 4. 解释正常路径

用四句话复述：输入是什么；谁拥有状态；程序允许了哪个动作；哪个环境结果证明任务完成。若只能复述 Agent 的自然语言，就还没有解释闭环。

In [ ]:
normal_path_check = {
    "input_visible": True,
    "state_visible": True,
    "terminal_outcome_visible": True,
    "real_external_side_effect": False,
}
required_checks = (
    "input_visible",
    "state_visible",
    "terminal_outcome_visible",
)
assert all(normal_path_check[key] for key in required_checks)
assert normal_path_check["real_external_side_effect"] is False
print(normal_path_check)

## 5. 失败路径：能力必须和边界一起出现

现在保持任务与实现不变，只改变一个关键条件。观察失败发生在提议、授权、执行、反馈还是终态判断。

In [ ]:
# [Cell 11: 失败路径 — 版本过期检测与乐观锁防覆盖]
try:
    from agent_course.courseware.public_examples import Channel, InboundRequest, ServiceCase, ServiceStore, StoreViolation
except ImportError:
    pass

transaction_store = ServiceStore()
case = transaction_store.create_case(ServiceCase('case-version', InboundRequest('request-version', 'tenant-au', Channel.EMAIL, 'customer@example.test', 'Version check', 'Please record this case.')))
updated = transaction_store.save_case(case, expected_version=case.version)
try:
    transaction_store.save_case(case, expected_version=case.version)
except StoreViolation as error:
    stale_write_reason = str(error)
    print({'stale_write_blocked': True, 'reason': stale_write_reason, 'current_version': updated.version})
else:
    raise AssertionError('stale write should be blocked')

## 6. 把正常与失败路径画在一起

下面的图直接读取本次运行产生的状态、结果与停止原因。先沿图复述“输入—动作—环境结果—终态”，再回到代码定位每一项由哪一行产生。图不是额外的装饰层。

In [ ]:
# [Cell 13: 泳道图可视化 (HTML Swimlane)]
from IPython.display import HTML, display

try:
    from agent_course.courseware.project_visuals import VisualStep, render_swimlane
except ImportError:
    import html
    from dataclasses import dataclass
    from typing import Sequence

    @dataclass(frozen=True, slots=True)
    class VisualStep:
        label: str
        detail: str
        status: str = "info"
        lane: str = "Runtime"

    def render_swimlane(*, title: str, question: str, lanes: Sequence[str], steps: Sequence[VisualStep], takeaway: str) -> str:
        lane_order = {lane: index for index, lane in enumerate(lanes)}
        columns = " ".join("minmax(130px,1fr)" for _ in lanes)
        headers = "".join(f'<div style="background:#eef1f4;padding:8px;text-align:center;font-weight:700;font-family:monospace;font-size:12px">{html.escape(lane)}</div>' for lane in lanes)
        cells = []
        for index, step in enumerate(steps):
            col_idx = lane_order.get(step.lane, 0) + 1
            color_map = {"passed": "#276749", "blocked": "#9b2c2c", "info": "#2457c5"}
            bg_map = {"passed": "#e7f4ec", "blocked": "#f8e7e7", "info": "#e8eefb"}
            c = color_map.get(step.status, "#2457c5")
            bg = bg_map.get(step.status, "#e8eefb")
            cells.append(
                f'<div style="grid-column:{col_idx};grid-row:{index + 2};background:{bg};border-left:4px solid {c};border:1px solid {c};border-radius:6px;padding:8px;display:flex;flex-direction:column;gap:4px">'
                f'<strong style="font-size:13px;color:#1f2933">{html.escape(step.label)}</strong>'
                f'<small style="color:#62707d;font-size:11px">{html.escape(step.detail)}</small></div>'
            )
        return (
            f'<div style="font-family:sans-serif;background:#f7f8fa;border:1px solid #cbd3da;border-radius:12px;padding:18px;margin:12px 0">'
            f'<h3 style="margin:0 0 6px">{html.escape(title)}</h3>'
            f'<p style="color:#62707d;font-size:13px;margin:0 0 12px"><strong>观察问题：</strong>{html.escape(question)}</p>'
            f'<div style="display:grid;grid-template-columns:{columns};gap:8px;overflow-x:auto;padding-bottom:8px">{headers}{"".join(cells)}</div>'
            f'<div style="border-left:4px solid #2457c5;padding:8px 12px;background:white;margin-top:12px;font-size:13px"><strong>读图结论：</strong>{html.escape(takeaway)}</div></div>'
        )

transaction_steps = (
    VisualStep("Case version", f"version={service_case.version}", "passed", lane="Store"),
    VisualStep("Supervisor approval", service_case.approval.approval_id, "passed", lane="Approval"),
    VisualStep("Create ticket", f"effects={len(service_platform.commands)}", "passed", lane="Platform"),
    VisualStep("Send test mail", f"receipts={len(service_mail.messages)}", "passed", lane="Mail"),
    VisualStep("Stale write", stale_write_reason, "blocked", lane="Store"),
    VisualStep("Business terminal state", service_outcome.stop_reason.value, "passed", lane="Runtime"),
)
display(
    HTML(
        render_swimlane(
            title="企业事务：局部成功必须收敛为一个业务终态",
            question="如果邮件失败，哪一个已经成功的动作不能重复？",
            lanes=("Store", "Approval", "Platform", "Mail", "Runtime"),
            steps=transaction_steps,
            takeaway="版本、批准、幂等 receipt 与 Outbox 共同决定如何恢复。",
        )
    )
)

## 6.1 不变量

不变量是无论模型怎样表达都不能被绕过的规则。它应由普通代码、存储约束、沙箱或独立监督器执行。

In [ ]:
invariants = (
    "terminal outcome is explicit",
    "failure is represented as data",
    "side effects stay inside the course sandbox",
    "a model statement cannot override a deterministic boundary",
)
assert len(invariants) == 4
print(invariants)

## 7. 连接到 Project

ServiceOps 事务里程碑：乐观并发、Outbox、幂等副作用、失败重试与对账。 本节代码不是另一本工具说明书，而是项目验收中的一个可重复证据。

## 8. 学生练习

完成下面的小函数。允许使用 Agent 帮助阅读报错和补充测试，但你必须能逐行解释判断条件，并亲自确认失败路径。

In [ ]:
def may_retry(idempotency_key: str, effect_exists: bool) -> str:
    # TODO: 已存在副作用时返回旧 receipt，不再次执行。
    raise NotImplementedError

assert may_retry("case-1:ticket", True) == "return_existing_receipt"
assert may_retry("case-1:ticket", False) == "execute:case-1:ticket"

## 9. 展示与验收

展示时只讲五件事：任务输入、关键状态、正常终态、失败终态、一个不能被模型绕过的不变量。练习时改变一项夹具，解释结果为什么改变。

## 10. 小结

回答开场问题：**怎样在部分失败时保持租户、版本、幂等、Outbox 与补偿的一致性？**

请用“输入 → 状态 → 动作 → 环境结果 → 停止原因”写一段不超过 120 字的复盘，并把失败断言迁移到 Project 测试。